# DDSM-ACGAN — benign vs malignant mammography augmentation on Colab

Applies the same ACGAN + VGG16 architecture from the CovidGAN reconstruction to **CBIS-DDSM mammography ROI patches**, classifying **benign vs malignant** rather than COVID vs Normal. Same pipeline shape: train an ACGAN on real ROIs → generate a synthetic pool → train the detection CNN with and without synthetic augmentation → compare.

**Before running:** `Runtime > Change runtime type > T4 GPU`. This notebook expects:
- A zipped folder of pre-extracted CBIS-DDSM ROI patches on your Drive, named like `P_<patient>_<LEFT|RIGHT>_<CC|MLO>_<mass|calcification>_<n>.png`.
- The four standard CBIS-DDSM case-description CSVs (mass/calc × train/test) on your Drive.

Repo: https://github.com/MayaHayat/generative_models — this notebook lives in `DDSM-ACGAN-Pytorch/`.

In [ ]:
!nvidia-smi

## 1. Mount Drive, clone the repo, install light dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/MayaHayat/generative_models.git
%cd generative_models/DDSM-ACGAN-Pytorch
!pip install -q scikit-learn matplotlib pillow

## 2. Unzip the ROI patches

Adjust the path to wherever your `ddsm_rois.zip` actually lives on Drive.

In [ ]:
ROI_ZIP = '/content/drive/MyDrive/Thesis/ddsm_rois.zip'
!unzip -q "{ROI_ZIP}" -d /content/ddsm_rois
!find /content/ddsm_rois -maxdepth 3 -type d

The unzip usually produces a top-level folder (e.g. `/content/ddsm_rois/rois`) containing the PNG files directly. If the listing above shows a different nesting, update `ROI_DIR` in the next cell to match.

In [ ]:
ROI_DIR = '/content/ddsm_rois/rois'
!ls {ROI_DIR} | head -5
!ls {ROI_DIR} | wc -l

## 3. Point at the CBIS-DDSM metadata CSVs

These carry the benign/malignant pathology labels and the official train/test split — adjust the folder if yours differs.

In [ ]:
CSV_DIR = '/content/drive/MyDrive/Thesis/tabular-dataset'
MASS_TRAIN_CSV = f'{CSV_DIR}/mass_case_description_train_set.csv'
MASS_TEST_CSV  = f'{CSV_DIR}/mass_case_description_test_set.csv'
CALC_TRAIN_CSV = f'{CSV_DIR}/calc_case_description_train_set.csv'
CALC_TEST_CSV  = f'{CSV_DIR}/calc_case_description_test_set.csv'

## 4. Build the manifest (join ROIs to pathology labels)

**Read the match-rate report this prints carefully.** If it's low, the CSV column names or ROI filenames don't match what this script assumes — check the printed sample of unmatched files before trusting anything downstream.

In [ ]:
!python prepare_ddsm.py --roi-dir {ROI_DIR} --mass-train-csv {MASS_TRAIN_CSV} --mass-test-csv {MASS_TEST_CSV} --calc-train-csv {CALC_TRAIN_CSV} --calc-test-csv {CALC_TEST_CSV} --out-dir data

## 5. Train the ACGAN — generator + discriminator

`GAN_EPOCHS = 2000` mirrors CovidGAN's setting. Start with something like 50 to sanity-check the whole pipeline before committing hours to it — and note how much smaller/larger your train set is than CovidGAN's 932 images, which changes how long each epoch actually takes.

**Checkpoints save directly to Drive** (`GAN_OUT_DIR`) rather than local Colab disk, since this is the one multi-hour step where losing progress to a session disconnect actually hurts. If a session drops mid-run, re-run cells 1-4 to get back to a working environment, then re-run this cell unchanged — it auto-detects and resumes from the latest checkpoint already on Drive instead of restarting.

In [ ]:
GAN_EPOCHS = 2000  # try e.g. 50 for a quick smoke test first
GAN_OUT_DIR = '/content/drive/MyDrive/ddsm_acgan_runs/gan'

import glob
checkpoints = sorted(glob.glob(f'{GAN_OUT_DIR}/checkpoints/ddsm_acgan_epoch*.pt'))
resume_flag = f'--resume {checkpoints[-1]}' if checkpoints else ''
print('resuming from:', checkpoints[-1] if checkpoints else '(fresh start, no checkpoint found)')

!python train_gan.py --manifest data/manifest.csv --out-dir {GAN_OUT_DIR} --epochs {GAN_EPOCHS} --batch-size 64 --lr 2e-4 --beta1 0.5 --sample-every 25 --checkpoint-every 100 {resume_flag}

Preview the most recent sample grid:

In [ ]:
from pathlib import Path
from IPython.display import Image, display

latest = sorted(Path(f'{GAN_OUT_DIR}/samples').glob('epoch_*.png'))[-1]
display(Image(filename=str(latest)))

## 6. Sample the trained generator into a synthetic pool

Check the class counts printed by `prepare_ddsm.py` above (step 4) and set these to roughly balance or oversample the minority class -- CBIS-DDSM is not perfectly balanced between benign and malignant, unlike the defaults below which assume a rough 600/600 split.

In [ ]:
!python generate_synthetic.py --checkpoint {GAN_OUT_DIR}/checkpoints/ddsm_acgan_final.pt --out-dir data/synthetic --n-benign 600 --n-malignant 600

## 7. CNN-AD — detection CNN trained on real ROIs only (baseline)

In [ ]:
!python train_classifier.py --manifest data/manifest.csv --mode ad --out-dir runs/cnn_ad --epochs 25 --batch-size 16 --lr 1e-3

## 8. CNN-SA — same CNN, trained on real + synthetic ROIs

In [ ]:
!python train_classifier.py --manifest data/manifest.csv --mode sa --synthetic-dir data/synthetic --out-dir runs/cnn_sa --epochs 25 --batch-size 16 --lr 1e-3

## 9. Compare CNN-AD vs CNN-SA

In [ ]:
print('=== CNN-AD (real ROIs only) ===')
print(Path('runs/cnn_ad/metrics.txt').read_text())
print()
print('=== CNN-SA (real + synthetic) ===')
print(Path('runs/cnn_sa/metrics.txt').read_text())

In [ ]:
from IPython.display import Image, display

print('CNN-AD confusion matrix')
display(Image(filename='runs/cnn_ad/confusion_matrix.png'))
print('CNN-SA confusion matrix')
display(Image(filename='runs/cnn_sa/confusion_matrix.png'))
print('PCA of penultimate-layer features (real vs. synthetic)')
display(Image(filename='runs/cnn_sa/pca.png'))

## 10. (Optional) Persist classifier results to Google Drive

The GAN checkpoints/samples are already on Drive (step 5 wrote there directly) — this just saves the classifier runs and manifest, which are cheap to regenerate but convenient to keep.

In [ ]:
!mkdir -p /content/drive/MyDrive/ddsm_acgan_runs
!cp -r runs/cnn_ad runs/cnn_sa /content/drive/MyDrive/ddsm_acgan_runs/
!cp data/manifest.csv /content/drive/MyDrive/ddsm_acgan_runs/